In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q22/annotated/checkpoints/pre_cell_2.pickle

In [ ]:
%%RecordEvent
%%time
### cell 2 ###

def_codes = ["13", "31", "23", "29", "30", "18", "17"]

# 1. extract country code and apply initial filters
def_codes = ["13", "31", "23", "29", "30", "18", "17"]
customer_filtered = (
    customer[
        ["C_CUSTKEY", "C_ACCTBAL", "C_PHONE"]
    ]
    .assign(CNTRYCODE=lambda df: df.C_PHONE.str.slice(0, 2))
)
customer_filtered = customer_filtered[
    (customer_filtered.C_ACCTBAL > 0.0)
    & customer_filtered.CNTRYCODE.isin(def_codes)
]

# 2. filter above-average balances
avg_balance = customer_filtered.C_ACCTBAL.mean()
customer_filtered = customer_filtered[customer_filtered.C_ACCTBAL > avg_balance]

# 3. anti-join via a left-anti merge (more efficient on GPU than isin)
customer_filtered = customer_filtered.merge(
    orders[["O_CUSTKEY"]],
    left_on="C_CUSTKEY", right_on="O_CUSTKEY",
    how="left_anti"
)

# 4. group and aggregate
total = (
    customer_filtered
    .groupby("CNTRYCODE")
    .agg({
        "C_CUSTKEY": "count",
        "C_ACCTBAL": "sum",
    })
    .reset_index()
    .rename(
        columns={
            "C_CUSTKEY": "NUMCUST",
            "C_ACCTBAL": "TOTACCTBAL",
        }
    )
    .sort_values("CNTRYCODE")
)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q22/rewritten/o4_mini_high/checkpoints/post_cell_2_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q22/opt_cell_exec_info_2_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[2], f)


In [ ]:
opt_output = Out.get(4)